# In Version two, we add additional filtering like remove descriptors with zero values ration >80%, variance <0.1, and Pearson correlation > 0.9

Load Data

In [1]:
import pandas as pd

df = pd.read_csv("Data_qsar.csv")

print(df.shape)
print(df.columns)
print(df.head())
print(df.info())

(2044, 4)
Index(['Unnamed: 0', 'ID', 'Smiles', 'pIC50'], dtype='object')
   Unnamed: 0  ID                                             Smiles  \
0           0   0  FC(F)Oc1c(CNC[C@@H](O)CC(=O)O)ccc(-c2c(C)c(-c3...   
1           1   1  Clc1c(-c2c(Cl)c(-c3cc(/C=C/c4ccc(OC)cc4)c(CNCC...   
2           2   2  Clc1c(-c2c(Cl)c(-c3cc(CCc4ccccc4)c(CNC[C@H]4NC...   
3           3   3  Clc1c(-c2c(O)c(-c3nc(OC)c(CNC[C@H]4NC(=O)CC4)c...   
4           4   4  Clc1c(-c2c(Cl)c(-c3nc(OC)c(CNC[C@H]4NC(=O)CC4)...   

       pIC50  
0  10.193820  
1   8.197705  
2   7.980178  
3   9.225483  
4   9.349692  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2044 entries, 0 to 2043
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  2044 non-null   int64  
 1   ID          2044 non-null   int64  
 2   Smiles      2044 non-null   object 
 3   pIC50       2044 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage

Remove Duplicate

In [2]:
print(df.duplicated().sum())
print(df.nunique())

0
Unnamed: 0    2044
ID            2044
Smiles        2022
pIC50          786
dtype: int64


See SMILES

In [4]:
print(df['Smiles'].nunique())

2022


In [5]:
df['pIC50'].describe()

,pIC50
count,2044.000000
mean,9.616435
std,0.609562
min,6.698536
25%,9.304521
50%,9.743524
75%,10.193820
max,10.292430


In [6]:
!pip install -q rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 42.9 MB/s eta 0:00:00


In [7]:
!pip install -q "numpy==2.0.2" "mordredcommunity[full]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 4.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors

from mordred import Calculator, descriptors

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor

# Keep important columns
df = df[["Smiles", "pIC50"]].copy()

# -----------------------------
# 2. Convert SMILES to molecules
# -----------------------------
df["Mol"] = df["Smiles"].apply(Chem.MolFromSmiles)

# remove invalid SMILES
invalid_count = df["Mol"].isna().sum()
print("Invalid SMILES:", invalid_count)

df = df[df["Mol"].notna()].reset_index(drop=True)
print("Shape after RDKit parsing:", df.shape)

# -----------------------------
# 3. RDKit descriptor calculation
# -----------------------------
def calc_rdkit_descriptors(mol):
    return Descriptors.CalcMolDescriptors(mol)

rdkit_desc = df["Mol"].apply(calc_rdkit_descriptors)
rdkit_desc_df = pd.DataFrame(rdkit_desc.tolist())

print("RDKit descriptor shape:", rdkit_desc_df.shape)

# -----------------------------
# 4. Mordred descriptor calculation
#    ignore_3D=True keeps it 2D based
# -----------------------------
calc = Calculator(descriptors, ignore_3D=True)
mordred_desc_df = calc.pandas(df["Mol"])

print("Mordred descriptor shape:", mordred_desc_df.shape)

# convert Mordred outputs to numeric where possible
mordred_desc_df = mordred_desc_df.apply(pd.to_numeric, errors="coerce")

# -----------------------------
# 5. Combine descriptors
# -----------------------------
X = pd.concat([rdkit_desc_df, mordred_desc_df], axis=1)
y = df["pIC50"].copy()

print("Combined descriptor shape before cleanup:", X.shape)

# -----------------------------
# 6. Cleanup
# -----------------------------
# replace inf with NaN
X = X.replace([np.inf, -np.inf], np.nan)

# drop columns with all missing values
X = X.dropna(axis=1, how="all")

# drop columns with zero variance
nunique = X.nunique(dropna=True)
X = X.loc[:, nunique > 1]

# optional: drop columns with too many missing values
missing_frac = X.isna().mean()
X = X.loc[:, missing_frac < 0.2]

# impute remaining missing values with median
X = X.fillna(X.median(numeric_only=True))

# make sure everything is numeric
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

print("Final feature matrix shape:", X.shape)

Additional Filtering

In [13]:

# 1. Remove descriptors with zero-value ratio > 80%
zero_ratio = (X == 0).sum() / len(X)
X = X.loc[:, zero_ratio <= 0.80]
print("After zero-value ratio filter:", X.shape)

# 2. Remove descriptors with variance < 0.1
from sklearn.feature_selection import VarianceThreshold

var_selector = VarianceThreshold(threshold=0.1)
X_var = var_selector.fit_transform(X)

selected_var_cols = X.columns[var_selector.get_support()]
X = pd.DataFrame(X_var, columns=selected_var_cols, index=X.index)
print("After variance filter:", X.shape)

# 3. Remove highly correlated descriptors (Pearson correlation > 0.9)
corr_matrix = X.corr().abs()

# keep only upper triangle
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
X = X.drop(columns=to_drop)

print("After correlation filter:", X.shape)

After zero-value ratio filter: (2044, 1285)
After variance filter: (2044, 729)
After correlation filter: (2044, 233)


In [14]:
# Make sure all column names are strings
X.columns = X.columns.astype(str)

# Check duplicate column names
dup_cols = X.columns[X.columns.duplicated()]
print("Number of duplicate columns:", dup_cols.shape[0])

if dup_cols.shape[0] > 0:
    print("Example duplicate columns:", dup_cols[:20].tolist())

# Remove duplicated columns, keep first occurrence
X = X.loc[:, ~X.columns.duplicated()]

print("Shape after removing duplicate columns:", X.shape)

Number of duplicate columns: 0
Shape after removing duplicate columns: (2044, 233)


In [16]:
# -----------------------------
# 7. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)



Train shape: (1635, 233)
Test shape: (409, 233)


Set up the Models

In [17]:
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting Machine": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}

Train and Evaluate

In [18]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
print(results_df)

                       Model      RMSE       MAE        R2
2                    XGBoost  0.351091  0.257756  0.657615
0              Random Forest  0.357508  0.264012  0.644985
1  Gradient Boosting Machine  0.361854  0.267096  0.636300


## Feature importance extraction

In [19]:
# after training Random Forest
importances = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

top_features = importances.head(50)["Feature"].tolist()

In [20]:
X_train_sel = X_train[top_features]
X_test_sel = X_test[top_features]

model.fit(X_train_sel, y_train)
pred = model.predict(X_test_sel)

In [30]:
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def evaluate_model(model, X_train, X_test, y_train, y_test, name="Model"):
    # fit
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # metrics
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"\n{name} Performance:")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    return {"Model": name, "MAE": mae, "RMSE": rmse,"R2": r2}

In [28]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

results = []

# Random Forest
rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
results.append(evaluate_model(rf, X_train, X_test, y_train, y_test, "Random Forest"))

# GBM
gbm = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
results.append(evaluate_model(gbm, X_train, X_test, y_train, y_test, "Gradient Boosting"))

# XGBoost
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)
results.append(evaluate_model(xgb, X_train, X_test, y_train, y_test, "XGBoost"))


Random Forest Performance:
MAE  : 0.2640
RMSE : 0.3575
R2   : 0.6450

Gradient Boosting Performance:
MAE  : 0.2671
RMSE : 0.3619
R2   : 0.6363

XGBoost Performance:
MAE  : 0.2578
RMSE : 0.3511
R2   : 0.6576


Convert to Table

In [32]:
import pandas as pd

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
print(results_df)

               Model        R2       MAE      RMSE
2            XGBoost  0.657615  0.257756  0.351091
0      Random Forest  0.644985  0.264012  0.357508
1  Gradient Boosting  0.636300  0.267096  0.361854


Compare before versus after feature selection

In [33]:
# using selected features (after your filtering / SHAP step)
results_sel = []

results_sel.append(evaluate_model(rf,  X_train_sel, X_test_sel, y_train, y_test, "RF - Selected"))
results_sel.append(evaluate_model(gbm, X_train_sel, X_test_sel, y_train, y_test, "GBM - Selected"))
results_sel.append(evaluate_model(xgb, X_train_sel, X_test_sel, y_train, y_test, "XGB - Selected"))

results_sel_df = pd.DataFrame(results_sel)
print(results_sel_df)


RF - Selected Performance:
MAE  : 0.2648
RMSE : 0.3605
R2   : 0.6391

GBM - Selected Performance:
MAE  : 0.2698
RMSE : 0.3675
R2   : 0.6248

XGB - Selected Performance:
MAE  : 0.2570
RMSE : 0.3558
R2   : 0.6484
            Model       MAE      RMSE        R2
0   RF - Selected  0.264794  0.360460  0.639098
1  GBM - Selected  0.269755  0.367534  0.624793
2  XGB - Selected  0.256971  0.355796  0.648376


Visualizing the results

In [ ]:
import matplotlib.pyplot as plt

y_pred = xgb.predict(X_test)

plt.scatter(y_test, y_pred)
plt.xlabel("Actual pIC50")
plt.ylabel("Predicted pIC50")
plt.title("Predicted vs Actual (XGBoost)")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()